# Fooocus Sunside — Ideal Colab

Фотореалізм · amateur / iPhone look · **краща анатомія тіла** · без цензури

### Як запустити
1. **Runtime → Change runtime type → GPU** → **Restart session**
2. У **КРОК 2** обери модель (default = **CyberRealistic**)
3. **Runtime → Run all** (перший раз **15–25 хв**)
4. Дочекайся **`App started successful`**
5. Відкрий **`https://….gradio.live`**

### Ideal stack (у всіх моделях)
| Шар | Що | Вага |
|-----|----|------|
| Body proportion | пропорції кінцівок | ~0.7 |
| Sufficient Nudity | NSFW / тіло | ~0.4 |
| SOAP | shot-on-phone | ~0.18 |
| **Face helper** | обличчя (баланс до тіла) | ~0.45–0.5 |
| Styles | **Sunside Iphone Selfie** + Semi Realistic | 1 phone-стиль |

💡 Якщо лице ще слабке → Advanced: face-helper **0.6**. Якщо тіло просіло → nudity **0.5** (не вище 0.6).

### Моделі
| Модель | Коли |
|--------|------|
| **CyberRealistic** (default) | чиста шкіра / «як фото» — зараз найкращий баланс |
| **epiCRealism Pure** | люди / тіло |
| **Juggernaut** | full body + universal |
| **RealVis** | лице / портрет |

⚠️ **Код -9** = OOM → Restart session → Run all. На T4 не чіпай RealCore.

💡 У Styles шукай **Sunside** · не вмикай 2+ phone-стилі разом · Fooocus V2 Expansion — OFF

In [ ]:
import os
import shutil
import subprocess
import sys
import time
import zipfile

os.chdir("/content")
os.makedirs("/content", exist_ok=True)

REPO = "/content/Fooocus"
REPO_URL = "https://github.com/sunsideaspect/foocus_sunside.git"
ZIP_URL = "https://github.com/sunsideaspect/foocus_sunside/archive/refs/heads/main.zip"

print("=" * 60)
print("КРОК 1/3 — Код foocus_sunside")
print("=" * 60)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pygit2==1.15.1"], check=True)


def remove_repo(path: str) -> None:
    if os.path.exists(path):
        subprocess.run(["rm", "-rf", path], check=False)


def repo_ok(path: str) -> bool:
    return os.path.isfile(os.path.join(path, "launch.py"))


def clone_with_git() -> bool:
    os.chdir("/content")
    remove_repo(REPO)
    print("→ git clone...")
    r = subprocess.run(
        ["git", "clone", "--depth", "1", "-b", "main", REPO_URL, REPO],
        capture_output=True,
        text=True,
        cwd="/content",
    )
    if r.returncode != 0:
        print("  git stderr:", (r.stderr or "").strip())
        return False
    return repo_ok(REPO)


def clone_with_zip() -> bool:
    os.chdir("/content")
    remove_repo(REPO)
    zip_path = "/content/foocus_sunside_main.zip"
    print("→ ZIP fallback...")
    r = subprocess.run(
        ["wget", "--progress=dot:giga", "-O", zip_path, ZIP_URL],
        cwd="/content",
    )
    if r.returncode != 0:
        return False
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall("/content")
    extracted = "/content/foocus_sunside-main"
    if not os.path.isdir(extracted):
        return False
    shutil.move(extracted, REPO)
    return repo_ok(REPO)


ok = False
for attempt in range(1, 4):
    if clone_with_git():
        ok = True
        print(f"✅ Код завантажено (git, спроба {attempt})")
        break
    print(f"  Повтор git через 8 с ({attempt}/3)...")
    time.sleep(8)

if not ok:
    ok = clone_with_zip()
    if ok:
        print("✅ Код завантажено (ZIP)")

if not ok:
    raise RuntimeError("Не вдалось завантажити. Runtime → Restart session → Run all")

os.chdir(REPO)
os.environ["TORCH_COMMAND"] = (
    "pip install torch torchvision torchaudio "
    "--index-url https://download.pytorch.org/whl/cu124"
)

print("\n" + "=" * 60)
print("Перевірка GPU")
print("=" * 60)
gpu_check = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
if gpu_check.returncode != 0 or "GPU" not in (gpu_check.stdout or ""):
    raise RuntimeError("❌ GPU не знайдено. Runtime → GPU → Restart session → Run all")
print("✅", (gpu_check.stdout or "").strip().split("\n")[0])
print("→ Далі: КРОК 2 (модель) + КРОК 3 (launch)")

In [ ]:
# @title ⚙️ КРОК 2 — Модель
# @markdown Ideal: body + nudity (м’якше) + **face-helper ON** + SOAP. Завантажується лише вибраний checkpoint.

model_choice = "CyberRealistic XL — clean photo (default, ~6.5 GB)"  # @param ["CyberRealistic XL — clean photo (default, ~6.5 GB)", "epiCRealism Pure — people / body (~7 GB)", "Juggernaut — full body (~7 GB)", "RealVis XL — face (~6.5 GB)"]

PRESET_BY_CHOICE = {
    "CyberRealistic XL — clean photo (default, ~6.5 GB)": "realistic_cyberrealistic_xl",
    "epiCRealism Pure — people / body (~7 GB)": "realistic_epicrealism_xl",
    "Juggernaut — full body (~7 GB)": "realistic_juggernaut_ragnarok",
    "RealVis XL — face (~6.5 GB)": "realistic_realvis_xl",
}
SELECTED_PRESET = PRESET_BY_CHOICE[model_choice]
print(f"✅ Preset: {SELECTED_PRESET}")
print("💡 LoRA: body 0.7 · nudity 0.4 · SOAP 0.18 · face-helper 0.45–0.5 ON")
print("💡 Лице слабке → face ↑0.6 | тіло слабке → nudity ↑0.5 (макс ~0.6)")
print("💡 Mirror selfie → Styles: Sunside Mirror Selfie (замість Iphone)")

In [ ]:
import os
import re
import stat
import subprocess
import sys

from IPython.display import HTML, display

if "SELECTED_PRESET" not in globals():
    SELECTED_PRESET = "realistic_cyberrealistic_xl"
    model_choice = "CyberRealistic (default — спочатку КРОК 2)"

os.chdir("/content")
os.makedirs("/content", exist_ok=True)

REPO = "/content/Fooocus"
if not os.path.isfile(os.path.join(REPO, "launch.py")):
    raise RuntimeError("Спочатку запусти КРОК 1 (клон репозиторію).")
os.chdir(REPO)

os.environ["GRADIO_ANALYTICS_ENABLED"] = "False"
os.environ["LAUNCH_LIVE_OUTPUT"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


def gpu_total_vram_mb() -> int:
    r = subprocess.run(
        ["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"],
        capture_output=True,
        text=True,
        check=True,
    )
    return int((r.stdout or "").strip().split("\n")[0])


vram_mb = gpu_total_vram_mb()
vram_flag = "--always-high-vram" if vram_mb >= 22000 else "--always-normal-vram"

print("=" * 60)
print("КРОК 3/3 — Запуск Fooocus (Ideal)")
print("=" * 60)
print(f"Preset: {SELECTED_PRESET}")
print(f"Model:  {model_choice}")
print(f"VRAM:   {vram_mb} MB → {vram_flag}")
if vram_mb < 16000:
    print("⚠️ T4: normal-vram. Якщо -9 → Restart session → Run all.")
print("\nЧекай: torch → моделі → Running on public URL → App started successful\n")


def _bar_html(pct: int, detail: str) -> str:
    pct = max(0, min(100, pct))
    safe = detail.replace("&", "&amp;").replace("<", "&lt;")
    return (
        f'<div style="margin:6px 0 10px">'
        f'<div style="background:#1e293b;border-radius:8px;height:26px;overflow:hidden;border:1px solid #334155">'
        f'<div style="width:{pct}%;height:100%;background:linear-gradient(90deg,#16a34a,#4ade80);'
        f'transition:width .4s"></div></div>'
        f'<div style="font:12px/1.4 monospace;color:#94a3b8;margin-top:4px">{pct}% — {safe}</div></div>'
    )


def _emit_progress(line: str, bar_holder: dict) -> bool:
    m = re.search(r"(\d+)%", line)
    if not m or not ("|" in line or "G/" in line or "M/" in line or "kB" in line):
        return False
    pct = int(m.group(1))
    html = _bar_html(pct, line.strip()[:140])
    if bar_holder.get("disp") is None:
        bar_holder["disp"] = display(HTML(html), display_id=True)
    else:
        bar_holder["disp"].update(HTML(html))
    return True


def run_live(argv, cwd=REPO, label=""):
    if label:
        print(label, flush=True)
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    env["LAUNCH_LIVE_OUTPUT"] = "1"
    p = subprocess.Popen(
        argv,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=env,
    )
    assert p.stdout is not None
    bar_holder: dict = {}
    partial = b""
    while True:
        chunk = p.stdout.read(1)
        if not chunk:
            if p.poll() is not None:
                break
            continue
        if chunk in (b"\r", b"\n"):
            line = partial.decode("utf-8", errors="ignore").strip()
            partial = b""
            if not line:
                continue
            if not _emit_progress(line, bar_holder):
                print(line, flush=True)
        else:
            partial += chunk
    tail = partial.decode("utf-8", errors="ignore").strip()
    if tail and not _emit_progress(tail, bar_holder):
        print(tail, flush=True)
    rc = p.wait()
    if rc != 0:
        if rc in (-9, 137):
            raise RuntimeError(
                "OOM (код -9). Runtime → Restart session → Run all. "
                "На T4: Juggernaut / Cyber / RealVis."
            )
        raise RuntimeError(f"Команда завершилась з кодом {rc}: {' '.join(argv)}")


print("→ A: numpy...")
run_live(
    [sys.executable, "-m", "pip", "install", "--force-reinstall", "numpy==1.26.4"],
    cwd="/content",
)
print("→ B: gradio...")
run_live(
    [sys.executable, "-m", "pip", "install",
     "starlette>=0.27.0,<1.0.0", "fastapi>=0.100.0,<0.115.0", "gradio==3.41.2"],
    cwd=REPO,
)


def gradio_pkg_dir() -> str:
    r = subprocess.run(
        [sys.executable, "-m", "pip", "show", "gradio"],
        capture_output=True,
        text=True,
        check=True,
    )
    for line in r.stdout.splitlines():
        if line.startswith("Location:"):
            return os.path.join(line.split(":", 1)[1].strip(), "gradio")
    raise RuntimeError("gradio not found after pip install")


print("→ C: frpc Share...")
frpc = os.path.join(gradio_pkg_dir(), "frpc_linux_amd64_v0.2")
if not (os.path.isfile(frpc) and os.path.getsize(frpc) > 1_000_000):
    subprocess.run(
        ["wget", "--progress=dot:giga", "-O", frpc,
         "https://cdn-media.huggingface.co/frpc-gradio-0.2/frpc_linux_amd64"],
        check=True,
    )
    os.chmod(frpc, stat.S_IRWXU)

launch_args = [
    sys.executable, "-u", "launch.py",
    "--preset", SELECTED_PRESET,
    "--disable-censor",
    "--disable-pro-mode",
    "--disable-preset-selection",
    "--share",
    vram_flag,
    "--disable-in-browser",
]

print("\n" + "=" * 60)
print("ЗАПУСК (лог у реальному часі)")
print("=" * 60 + "\n")

os.chdir(REPO)
run_live(launch_args, cwd=REPO)
print("\n✅ Сесію зупинено (нормально, якщо ти сам вимкнув).")